```markdown
# مشروع اكتشاف الوجوه وتقدير العمر باستخدام OpenCV
تستخدم هذه المفكرة نماذج Pre-trained (Caffe و TensorFlow) لاكتشاف الوجوه وتوقع الفئة العمرية.
```

In [ ]:
import cv2
import numpy as np
import os

# تعريف مسارات الملفات
face_proto = "opencv_face_detector.pbtxt"
face_model = "opencv_face_detector_uint8.pb"
age_proto = "age_deploy.prototxt"
age_model = "age_net.caffemodel"

# التحقق من وجود الملفات
files = [face_proto, face_model, age_proto, age_model]
missing_files = [f for f in files if not os.path.exists(f)]

if missing_files:
    print(f"خطأ: الملفات التالية مفقودة: {missing_files}")
    print("يرجى التأكد من رفع ملفات النماذج إلى مجلد الملفات في Colab.")
else:
    # تحميل النماذج
    face_net = cv2.dnn.readNetFromTensorflow(face_model, face_proto)
    age_net = cv2.dnn.readNetFromCaffe(age_proto, age_model)
    print("✅ تم تحميل نماذج اكتشاف الوجه والعمر بنجاح.")

# الثوابت
MODEL_MEAN_VALUES = (78.4263377603, 87.7689143744, 114.895847746)
age_list = ['(0-2)', '(4-6)', '(8-12)', '(15-20)', '(25-32)', '(38-43)', '(48-53)', '(60-100)']

```markdown
### تعريف الدوال الأساسية
```

In [ ]:
def detect_faces(net, frame, confidence_threshold=0.7):
    frame_height = frame.shape[0]
    frame_width = frame.shape[1]
    blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), [104, 117, 123], False, False)
    net.setInput(blob)
    detections = net.forward()
    face_boxes = []

    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]
        if confidence > confidence_threshold:
            x1 = int(detections[0, 0, i, 3] * frame_width)
            y1 = int(detections[0, 0, i, 4] * frame_height)
            x2 = int(detections[0, 0, i, 5] * frame_width)
            y2 = int(detections[0, 0, i, 6] * frame_height)
            face_boxes.append([x1, y1, x2, y2])
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    return frame, face_boxes

def predict_age(face, net):
    blob = cv2.dnn.blobFromImage(face, 1.0, (227, 227), MODEL_MEAN_VALUES, swapRB=False)
    net.setInput(blob)
    age_preds = net.forward()
    age = age_list[age_preds[0].argmax()]
    return age

```markdown
### تشغيل الكاميرا
ملاحظة: إذا كنت تستخدم Colab، ستحتاج لاستخدام JavaScript Snippet للوصول إلى الكاميرا.
```

In [ ]:
def run_detection(frame):
    """دالة لمعالجة إطار واحد وعرض النتائج"""
    frame, face_boxes = detect_faces(face_net, frame)
    for (x1, y1, x2, y2) in face_boxes:
        face = frame[max(0, y1-20):min(y2+20, frame.shape[0]-1),
                     max(0, x1-20):min(x2+20, frame.shape[1]-1)]

        if face.shape[0] > 0 and face.shape[1] > 0:
            age = predict_age(face, age_net)
            cv2.putText(frame, f"Age: {age}", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    return frame

# ملاحظة: استدعاء الكاميرا التقليدي
# if __name__ == "__main__":
#     run_camera()